# 03 — Distribution Fitting Diagnostics & Model Validation

**Purpose:** Notebook 02 fitted three innovation distributions **per-window** inside the rolling loop:

| Distribution | Parameters fitted per window |
|---|---|
| **NIG** | $\alpha, \beta, \mu, \delta$ via MLE | 
| **Student-T** | $\nu$ (dof) via MLE with `floc=0, fscale=1` | 
| **Gaussian** | None ($\Phi$ is parameter-free after GARCH standardisation) | 

All three PIT series are computed on the same out-of-sample standardised innovation $z_{t+1} = (r_{t+1} - \mu_{t+1})/\sigma_{t+1}$, making the comparison **apples-to-apples**.

**Validation principle (Kaufman slide 35):** If the model is correct, $u_t = F_t(z_t) \sim U[0,1]$.

In [4]:
# Imports

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from scipy import stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "iframe"

import warnings
warnings.filterwarnings("ignore")

from src.data_utils import load_processed, save_processed
from src.nig import NIGParams, nig_pdf, nig_cdf
from src.assessment import pit_qq, pit_ks_test, anderson_darling_pit

print("Imports OK")

Imports OK


In [5]:
# Load rolling results from notebook 02

innovations = load_processed("innovations.parquet")
pit_nig_df  = load_processed("pit_nig.parquet")
pit_t_df    = load_processed("pit_t.parquet")
pit_gauss_df = load_processed("pit_gauss.parquet")

TICKERS = list(innovations.columns)

# Load full rolling results per asset
all_results = {}
for ticker in TICKERS:
    safe = ticker.replace("^", "").replace("=", "_")
    all_results[ticker] = load_processed(f"rolling_{safe}.parquet")

print(f"Loaded: {len(TICKERS)} assets, {pit_nig_df.shape[0]} prediction dates")
print(f"Tickers: {TICKERS}")
print(f"Date range: {pit_nig_df.index[0].date()} → {pit_nig_df.index[-1].date()}")
print(f"PIT series: NIG {pit_nig_df.shape}, T {pit_t_df.shape}, Gauss {pit_gauss_df.shape}")


Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\innovations.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\pit_nig.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\pit_t.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\pit_gauss.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-

---
## 1. Per-Window Parameter Summary

In [6]:
# NIG and Student-T parameter summary across all windows

rows = []
for ticker in TICKERS:
    df = all_results[ticker]
    nig_conv = df["nig_converged"].sum()
    rows.append({
        "Ticker": ticker,
        "NIG α (med)": round(df["nig_alpha"].median(), 4),
        "NIG β (med)": round(df["nig_beta"].median(), 4),
        "NIG δ (med)": round(df["nig_delta"].median(), 4),
        "NIG α (std)": round(df["nig_alpha"].std(), 4),
        "T dof (med)": round(df["t_dof"].median(), 2),
        "T dof (std)": round(df["t_dof"].std(), 2),
        "NIG conv": f"{nig_conv}/{len(df)}",
    })

param_summary = pd.DataFrame(rows).set_index("Ticker")
print("Per-Window Parameter Summary\n")
display(param_summary)

print("\nSmaller NIG α → heavier tails.  Smaller T dof → heavier tails.")
print("Non-zero std confirms parameters vary across windows (per-window fitting working).")

Per-Window Parameter Summary



,NIG α (med),NIG β (med),NIG δ (med),NIG α (std),T dof (med),T dof (std),NIG conv
Ticker,,,,,,,
AAPL,1.5075,-0.0353,1.5112,0.7883,15.15,2.078000e+01,954/1000
EURUSD=X,1.9794,-0.0279,1.8866,12.0765,27.73,7.132001e+12,870/1000
GLD,1.8514,-0.2066,1.7406,10.8009,24.42,2.221000e+01,969/1000
TIP,1.9024,0.1040,1.7971,14.4602,21.08,8.210000e+00,937/1000
^GSPC,4.0197,-0.4264,3.1502,20.1550,49.93,1.220102e+13,1000/1000



Smaller NIG α → heavier tails.  Smaller T dof → heavier tails.
Non-zero std confirms parameters vary across windows (per-window fitting working).


In [7]:
# NIG and Student-T parameter evolution over time

colors_list = px.colors.qualitative.Plotly

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=["NIG α (tail heaviness)", "NIG β (asymmetry)", "Student-T dof"],
    vertical_spacing=0.08,
)

for i, ticker in enumerate(TICKERS):
    df = all_results[ticker]
    fig.add_trace(go.Scatter(
        x=df.index, y=df["nig_alpha"], mode="lines",
        name=ticker, showlegend=True,
        line=dict(color=colors_list[i], width=0.8),
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=df.index, y=df["nig_beta"], mode="lines",
        name=ticker, showlegend=False,
        line=dict(color=colors_list[i], width=0.8),
    ), row=2, col=1)
    fig.add_trace(go.Scatter(
        x=df.index, y=df["t_dof"], mode="lines",
        name=ticker, showlegend=False,
        line=dict(color=colors_list[i], width=0.8),
    ), row=3, col=1)

fig.update_layout(
    title="Per-Window Parameters Over Time: NIG and Student-T",
    height=700, template="plotly_white", hovermode="x unified",
)
fig.write_html("../outputs/figures/03_params_evolution.html")
fig.show()
print("Figure saved")


Figure saved


---
## 2. PDF Comparison (Illustrative)

Using **median** parameters across windows as a representative snapshot. This is for visualisation — the formal tests use the per-window PIT values.

In [8]:
# PDF comparison: NIG vs Student-T vs Gaussian

x_grid = np.linspace(-7, 7, 600)
gaussian_ref = stats.norm.pdf(x_grid)

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=TICKERS + [""],
    vertical_spacing=0.12, horizontal_spacing=0.06,
)

for idx, ticker in enumerate(TICKERS):
    row, col = idx // 3 + 1, idx % 3 + 1
    innov = innovations[ticker].dropna().values
    df_r = all_results[ticker]

    # Median NIG
    nig_rep = NIGParams(
        alpha=df_r["nig_alpha"].median(), beta=df_r["nig_beta"].median(),
        mu=df_r["nig_mu"].median(), delta=df_r["nig_delta"].median(),
    )
    nig_y = nig_pdf(x_grid, nig_rep)

    # Median Student-T
    t_dof_med = df_r["t_dof"].median()
    t_y = stats.t.pdf(x_grid, t_dof_med, loc=0, scale=1)

    fig.add_trace(go.Histogram(
        x=innov, nbinsx=80, histnorm="probability density",
        name="Empirical" if idx == 0 else None, showlegend=(idx == 0),
        marker_color="lightgrey", opacity=0.6,
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=x_grid, y=nig_y, mode="lines",
        name="NIG" if idx == 0 else None, showlegend=(idx == 0),
        line=dict(color="#d62728", width=2),
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=x_grid, y=t_y, mode="lines",
        name="Student-T" if idx == 0 else None, showlegend=(idx == 0),
        line=dict(color="#ff7f0e", width=1.5, dash="dash"),
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=x_grid, y=gaussian_ref, mode="lines",
        name="Gaussian" if idx == 0 else None, showlegend=(idx == 0),
        line=dict(color="black", width=1, dash="dot"),
    ), row=row, col=col)

fig.update_layout(
    title="Innovation Distributions: NIG vs Student-T vs Gaussian (median params)",
    height=600, template="plotly_white",
)
fig.update_xaxes(range=[-6, 6])
fig.write_html("../outputs/figures/03_pdf_comparison.html")
fig.show()
print("Figure saved")

Figure saved


---
## 3. PIT-Based Model Validation

All three PIT series were computed **per-window** inside the rolling loop (notebook 02):

- **NIG PIT:** $u_t = F_{\text{NIG}_t}(z_t)$ with per-window $\text{NIG}(\alpha_t, \beta_t, \mu_t, \delta_t)$
- **Student-T PIT:** $u_t = F_{T_t}(z_t; \nu_t)$ with per-window dof $\nu_t$ (Kaufman slide 29: `scipy.stats.t.fit(data, floc=0, fscale=1)`)
- **Gaussian PIT:** $u_t = \Phi(z_t)$ — parameter-free, valid as-is

This is a fair **apples-to-apples** comparison.

In [10]:
# PIT histograms: NIG vs Student-T vs Gaussian

fig = make_subplots(
    rows=len(TICKERS), cols=3,
    column_titles=["NIG PIT", "Student-T PIT", "Gaussian PIT"],
    row_titles=TICKERS,
    vertical_spacing=0.04, horizontal_spacing=0.05,
)

pit_dfs = {"NIG": pit_nig_df, "T": pit_t_df, "Gauss": pit_gauss_df}
pit_colors = {"NIG": "#d62728", "T": "#ff7f0e", "Gauss": "#1f77b4"}

for i, ticker in enumerate(TICKERS):
    for j, (label, pdf) in enumerate(pit_dfs.items()):
        pit = pdf[ticker].dropna().values
        fig.add_trace(go.Histogram(
            x=pit, nbinsx=20, histnorm="probability density",
            marker_color=pit_colors[label], opacity=0.7,
            showlegend=False,
        ), row=i+1, col=j+1)
        fig.add_hline(y=1.0, line_dash="dash", line_color="black",
                      line_width=0.8, row=i+1, col=j+1)
        fig.update_xaxes(range=[0, 1], row=i+1, col=j+1)

fig.update_layout(
    title="PIT Histograms ",
    height=200 * len(TICKERS), template="plotly_white",
)
fig.write_html("../outputs/figures/03_pit_histograms.html")
fig.show()
print("Figure saved")


Figure saved


In [15]:
# PIT QQ plots: NIG vs Student-T vs Gaussian 

fig = make_subplots(
    rows=len(TICKERS), cols=3,
    column_titles=["NIG PIT QQ", "Student-T PIT QQ", "Gaussian PIT QQ"],
    row_titles=TICKERS,
    vertical_spacing=0.06, horizontal_spacing=0.08,
)

for i, ticker in enumerate(TICKERS):
    for j, (label, pdf) in enumerate(pit_dfs.items()):
        pit = pdf[ticker].dropna().values
        emp_q, mod_q = pit_qq(pit)

        fig.add_trace(go.Scatter(
            x=emp_q, y=mod_q, mode="markers",
            marker=dict(color=pit_colors[label], size=2),
            showlegend=False,
        ), row=i+1, col=j+1)
        fig.add_trace(go.Scatter(
            x=[0,1], y=[0,1], mode="lines", showlegend=False,
            line=dict(color="black", width=0.8, dash="dash"),
        ), row=i+1, col=j+1)
        fig.update_xaxes(range=[0,1], row=i+1, col=j+1)
        fig.update_yaxes(range=[0,1], row=i+1, col=j+1)
        fig.update_xaxes(title_text="Empirical quantile",
                         title_font=dict(size=9, color="rgba(0,0,0,0.4)"),
                         title_standoff=2, range=[0,1], row=i+1, col=j+1,
        )
        fig.update_yaxes(title_text="Model quantile",
                         title_font=dict(size=9, color="rgba(0,0,0,0.4)"),
                         title_standoff=2,
                         range=[0,1], row=i+1, col=j+1,             
        )
        
fig.update_layout(
    title="PIT QQ Plots Against Uniform(0,1)",
    height=200 * len(TICKERS), template="plotly_white",
)

fig.write_html("../outputs/figures/03_pit_qq.html")
fig.show()
print("Figure saved")

Figure saved


---
## 4. Goodness-of-Fit Tests

In [16]:
# KS and Anderson-Darling on PIT values: NIG vs Student-T vs Gaussian
# All three computed per-window — fair comparison.

gof_rows = []
for ticker in TICKERS:
    pit_n = pit_nig_df[ticker].dropna().values
    pit_t = pit_t_df[ticker].dropna().values
    pit_g = pit_gauss_df[ticker].dropna().values

    ks_n = pit_ks_test(pit_n)
    ks_t = pit_ks_test(pit_t)
    ks_g = pit_ks_test(pit_g)

    ad_n = anderson_darling_pit(pit_n)
    ad_t = anderson_darling_pit(pit_t)
    ad_g = anderson_darling_pit(pit_g)

    # Determine winner
    ks_vals = {"NIG": ks_n["statistic"], "T": ks_t["statistic"], "Gauss": ks_g["statistic"]}
    ad_vals = {"NIG": ad_n, "T": ad_t, "Gauss": ad_g}
    ks_winner = min(ks_vals, key=ks_vals.get)
    ad_winner = min(ad_vals, key=ad_vals.get)

    ks_nig_vs_gauss = (1 - ks_n["statistic"] / max(ks_g["statistic"], 1e-10)) * 100

    gof_rows.append({
        "Ticker": ticker,
        "NIG KS": ks_n["statistic"], "NIG KS p": ks_n["pvalue"],
        "T KS": ks_t["statistic"], "T KS p": ks_t["pvalue"],
        "Gauss KS": ks_g["statistic"], "Gauss KS p": ks_g["pvalue"],
        "NIG AD": round(ad_n, 2),
        "T AD": round(ad_t, 2),
        "Gauss AD": round(ad_g, 2),
        "Best (KS)": ks_winner,
        "Best (AD)": ad_winner,
        "NIG vs Gauss (KS)": f"{ks_nig_vs_gauss:.1f}%",
    })

gof_df = pd.DataFrame(gof_rows).set_index("Ticker")
print("Goodness-of-Fit: PIT Values Against Uniform(0,1)")
print("All three distributions fitted per-window — fair comparison\n")
display(gof_df)

print("\nKS: smaller statistic = closer to Uniform. p > 0.05 = model adequate.")
print("AD: smaller = better, with extra weight on tail deviations.")
print("NIG vs Gauss (KS): % improvement in KS statistic from Gaussian to NIG.")

Goodness-of-Fit: PIT Values Against Uniform(0,1)
All three distributions fitted per-window — fair comparison



,NIG KS,NIG KS p,T KS,T KS p,Gauss KS,Gauss KS p,NIG AD,T AD,Gauss AD,Best (KS),Best (AD),NIG vs Gauss (KS)
Ticker,,,,,,,,,,,,
AAPL,0.0164,0.9467,0.0517,0.0091,0.0490,0.0158,0.51,3.66,3.40,NIG,NIG,66.5%
EURUSD=X,0.0139,0.9887,0.0387,0.0981,0.0343,0.1847,0.36,1.65,1.47,NIG,NIG,59.5%
GLD,0.0227,0.6726,0.0402,0.0764,0.0389,0.0949,1.10,1.90,2.10,NIG,NIG,41.6%
TIP,0.0295,0.3411,0.0423,0.0538,0.0368,0.1294,0.83,3.50,2.61,NIG,NIG,19.8%
^GSPC,0.0230,0.6563,0.0323,0.2426,0.0289,0.3667,0.98,2.56,2.63,NIG,NIG,20.4%



KS: smaller statistic = closer to Uniform. p > 0.05 = model adequate.
AD: smaller = better, with extra weight on tail deviations.
NIG vs Gauss (KS): % improvement in KS statistic from Gaussian to NIG.


---

In [17]:
# Save results

save_processed(gof_df.reset_index(), "gof_pit_results.parquet")
save_processed(param_summary.reset_index(), "param_summary.parquet")
print("Saved: gof_pit_results.parquet, param_summary.parquet")

Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\gof_pit_results.parquet
Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\param_summary.parquet
Saved: gof_pit_results.parquet, param_summary.parquet


In [18]:
# Verification

print("Notebook 03 — Summary\n")
print(f"Assets: {len(TICKERS)}")
print(f"Predictions: {pit_nig_df.shape[0]} per asset")

print("\nPIT KS results (all per-window):")
print(f"{'Ticker':12s}  {'NIG KS':>8s}  {'T KS':>8s}  {'Gauss KS':>8s}  Best")
for _, row in gof_df.iterrows():
    print(f"{row.name:12s}  {row['NIG KS']:8.4f}  {row['T KS']:8.4f}  "
          f"{row['Gauss KS']:8.4f}  {row['Best (KS)']}")

print("\nAll comparisons are apples-to-apples (per-window fitting).")
print("Proceed to notebook 04 (risk measures).")

Notebook 03 — Summary

Assets: 5
Predictions: 1000 per asset

PIT KS results (all per-window):
Ticker          NIG KS      T KS  Gauss KS  Best
AAPL            0.0164    0.0517    0.0490  NIG
EURUSD=X        0.0139    0.0387    0.0343  NIG
GLD             0.0227    0.0402    0.0389  NIG
TIP             0.0295    0.0423    0.0368  NIG
^GSPC           0.0230    0.0323    0.0289  NIG

All comparisons are apples-to-apples (per-window fitting).
Proceed to notebook 04 (risk measures).


---